# Loading dataset and preprocessing

In [1]:
import pandas as pd
df = pd.read_csv('Fake_Real_Data.csv')
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [2]:
df['label'].value_counts()

,count
label,
Fake,5000
Real,4900


In [3]:
df['label_num'] = df['label'].map({'Fake':0, 'Real':1})
df.head(3)

,Text,label,label_num
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0
1,U.S. conservative leader optimistic of common ...,Real,1
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1


In [4]:
# Preprocess function
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger", "ner"])

def preprocess(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
        results.append(" ".join(tokens))
    return results

In [5]:
df["preprocessed_txt"] = preprocess(df["Text"])

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [6]:
df.head(3)

,Text,label,label_num,preprocessed_txt
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0,trump surrogate brutally stabs pathetic vide...
1,U.S. conservative leader optimistic of common ...,Real,1,u.s. conservative leader optimistic common gro...
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1,trump proposes u.s. tax overhaul stirs concern...


In [7]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["preprocessed_txt"]
y = df["label_num"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Applying BoW and naive bayes, random forest

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9787878787878788

In [9]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       973
           1       0.98      0.98      0.98      1007

    accuracy                           0.98      1980
   macro avg       0.98      0.98      0.98      1980
weighted avg       0.98      0.98      0.98      1980



In [10]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('rf', RandomForestClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.997979797979798

In [11]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       973
           1       1.00      1.00      1.00      1007

    accuracy                           1.00      1980
   macro avg       1.00      1.00      1.00      1980
weighted avg       1.00      1.00      1.00      1980



# Applying n-grams and naive bayes, random forest

In [ ]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(2,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9904040404040404

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       973
           1       0.98      1.00      0.99      1007

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980



In [ ]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(2,2))),
    ('rf', RandomForestClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9914141414141414

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       973
           1       0.98      1.00      0.99      1007

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980



# Applying TF-IDF and Naive Bayes, Random forest

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9646464646464646

In [13]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.96      0.96       973
           1       0.96      0.97      0.97      1007

    accuracy                           0.96      1980
   macro avg       0.96      0.96      0.96      1980
weighted avg       0.96      0.96      0.96      1980



In [14]:
clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('rf', RandomForestClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9954545454545455

In [15]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       973
           1       1.00      0.99      1.00      1007

    accuracy                           1.00      1980
   macro avg       1.00      1.00      1.00      1980
weighted avg       1.00      1.00      1.00      1980



### Making predictions

In [16]:
X_test[:3]

,preprocessed_txt
8432,trump involvement houston chemical plant exp...
5680,u.s. senate votes near unanimously russia iran...
4767,white male broader bureaucracy mirrors trump c...


In [17]:
y_test[:3]

,label_num
8432,0
5680,1
4767,1


In [18]:
y_pred[:3]

array([0, 1, 1])

# Using Spacy word vectors

In [22]:
!python -m spacy download en_core_web_lg

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl (400.7 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [23]:
import spacy
nlp = spacy.load("en_core_web_lg", disable=["parser", "ner", "tagger"])

In [24]:
vectors = []
for doc in nlp.pipe(df["preprocessed_txt"], batch_size=1000):
    vectors.append(doc.vector)

df["vector"] = vectors

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [25]:
df.head()

,Text,label,label_num,preprocessed_txt,vector
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0,trump surrogate brutally stabs pathetic vide...,"[-0.21843824, 0.12057698, -0.004211821, 0.0167..."
1,U.S. conservative leader optimistic of common ...,Real,1,u.s. conservative leader optimistic common gro...,"[-0.02786422, 0.1467866, 0.042411074, 0.052750..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1,trump proposes u.s. tax overhaul stirs concern...,"[-0.22834532, 0.14353442, 0.109243825, -0.0849..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0,court forces ohio allow millions illegally p...,"[-0.06114301, 0.06436468, 0.1669973, -0.027036..."
4,Democrats say Trump agrees to work on immigrat...,Real,1,democrats trump agrees work immigration bill w...,"[-0.087871626, 0.035694152, 0.11290749, 0.0155..."


In [26]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["vector"]
y = df["label_num"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
import numpy as np
X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [33]:
# Applying Naive bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.945959595959596

In [38]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.94      0.94       973
           1       0.95      0.95      0.95      1007

    accuracy                           0.95      1980
   macro avg       0.95      0.95      0.95      1980
weighted avg       0.95      0.95      0.95      1980



In [37]:
y_pred

array([0, 1, 1, ..., 0, 1, 1])

In [39]:
# Applying KNN
from sklearn.neighbors import KNeighborsClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, metric='euclidean'))
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9681818181818181

In [40]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.97       973
           1       0.95      0.98      0.97      1007

    accuracy                           0.97      1980
   macro avg       0.97      0.97      0.97      1980
weighted avg       0.97      0.97      0.97      1980



# Using gensim word2vec

In [41]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.7 MB/s eta 0:00:00


In [43]:
import gensim.downloader as api
model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [44]:
vectors = []
for text in df["preprocessed_txt"]:
    tokens = text.split()
    vec = model.get_mean_vector(tokens)
    vectors.append(vec)

df["vector"] = vectors

In [45]:
df.head()

,Text,label,label_num,preprocessed_txt,vector
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0,trump surrogate brutally stabs pathetic vide...,"[-0.0062560537, 0.03445515, 0.0006646663, 0.04..."
1,U.S. conservative leader optimistic of common ...,Real,1,u.s. conservative leader optimistic common gro...,"[0.0067335507, 0.0069558886, 0.014647508, 0.01..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1,trump proposes u.s. tax overhaul stirs concern...,"[0.01651386, 0.011694943, 0.0022757594, 0.0429..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0,court forces ohio allow millions illegally p...,"[0.004882626, 0.02290615, 0.01145851, 0.025004..."
4,Democrats say Trump agrees to work on immigrat...,Real,1,democrats trump agrees work immigration bill w...,"[-0.0040321397, 0.017716428, 0.014590898, 0.02..."


In [46]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["vector"]
y = df["label_num"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [47]:
import numpy as np
X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [48]:
# Applying Naive bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9338383838383838

In [49]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93       973
           1       0.93      0.95      0.94      1007

    accuracy                           0.93      1980
   macro avg       0.93      0.93      0.93      1980
weighted avg       0.93      0.93      0.93      1980



In [50]:
# Applying gradient boosting classifier
from sklearn.ensemble import GradientBoostingClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('gbc', GradientBoostingClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9797979797979798

In [51]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       973
           1       0.98      0.98      0.98      1007

    accuracy                           0.98      1980
   macro avg       0.98      0.98      0.98      1980
weighted avg       0.98      0.98      0.98      1980

